In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

In [2]:

executor = ThreadPoolExecutor(max_workers=25)

In [3]:
import os
import shutil

def candy(func):
    func_name = func.__name__

    def wrapper(*args, **kwargs):
        # Create directories if not exist
        if not os.path.exists(func_name):
            os.makedirs(func_name)
        if not os.path.exists(os.path.join(func_name, 'old')):
            os.makedirs(os.path.join(func_name, 'old'))

        # Modify output function to redirect file output
        def output_func(filename, data):
            # If file already exists, rename and move it
            target_path = os.path.join(func_name, filename)
            if os.path.exists(target_path):
                old_files = sorted([f for f in os.listdir(os.path.join(func_name, 'old')) if f.startswith(filename)])
                if old_files:
                    last_file = old_files[-1]
                    last_index = int(last_file.split('-')[-1])
                else:
                    last_index = 0
                os.rename(target_path, os.path.join(func_name, 'old', filename + '-' + str(last_index + 1)))
            
            with open(target_path, 'w') as f:
                f.write(data)

        # Run the function with the new output_func as argument
        func(*args, output_func=output_func, **kwargs)

    return wrapper


In [4]:
def scriptToEPL(previous=None, current=None, next=None):
    if not current: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Analyze this phrase for Ethos (E), Pathos (P), and Logos (L) from 0.0 - 9.9: '{current}'. Return a compact dictionary format like this: 'E: x.x, P: y.y, L: z.z, E_Explanation: "", P_Explanation: "", L_Explanation: "". Explanations must be 1 sentence long."
    
    if previous:
        content = f"Given the previous sentence: '{previous}', " + content
    if next:
        content = content + f" Also, consider the following sentence: '{next}'."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt

In [5]:
def process_lines(lines):
    """
    Processes the transcript lines into speaker turns.
    Each speaker turn starts with a line that has a colon ':'.
    """
    speaker_turns = []
    current_turn = []

    for line in lines:
        if ':' in line:  # This line is the start of a new speaker turn
            speaker_turns.append(' '.join(current_turn))  # Add the current turn to the turns list
            current_turn = [line]  # Start a new turn with this line
        else:
            current_turn.append(line)  # This line is part of the current turn

    speaker_turns.append(' '.join(current_turn))  # Add the last turn

    # The first element is empty, so we skip it
    speaker_turns = speaker_turns[1:]

    # Write speaker turns to file
    with open("speaker_turns.txt", "w") as file:
        for i, turn in enumerate(speaker_turns):
            file.write(f"Speaker turn {i+1}:\n{turn}\n\n")

    return speaker_turns

In [6]:
def get_turns(speaker_turns, i):
    """
    Returns the previous, current, and next turns given a list of speaker turns and an index.
    """
    prev_turn = speaker_turns[i - 1] if i > 0 else ''  # If this is the first turn, there is no previous turn
    curr_turn = speaker_turns[i]
    next_turn = speaker_turns[i + 1] if i < len(speaker_turns) - 1 else ''  # If this is the last turn, there is no next turn

    return prev_turn, curr_turn, next_turn

In [7]:
def threadlock(i, speaker_turns, output_file_full, output_file_content, lock1, lock2, key):
    # Get previous, current and next turn
    prev_turn, curr_turn, next_turn = get_turns(speaker_turns, i)
    
    # Set the API key
    openai.api_key = key
    
    # Call OpenAI API and get response
    response = scriptToEPL(prev_turn, curr_turn, next_turn)
    
    # Write complete response to JSON file
    lock1.acquire()
    try:
        with open(output_file_full, 'a') as outfile:
            json.dump(response, outfile)
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock1.release()

    # Extract 'content' field and write to second JSON file
    content = response['choices'][0]['message']['content']
    
    # Include the turns that were passed to GPT in the content
    content_with_turns = {
        "prev_turn": prev_turn,
        "curr_turn": curr_turn,
        "next_turn": next_turn,
        "content": content
    }
    
    lock2.acquire()
    try:
        with open(output_file_content, 'a') as outfile:
            json.dump(content_with_turns, outfile)  # Write the content with turns instead of just content
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock2.release()

In [8]:
def epl(input_file, output_file_full, output_file_content, startline=None, endline=None, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startline is not None and startturn is not None:
        raise ValueError("Parameters 'startline' and 'startturn' are mutually exclusive. Use only 'startline' to specify the line number to start analysis, or 'startturn' to specify the speaker turn number to start analysis.")
    if endline is not None and endturn is not None:
        raise ValueError("Parameters 'endline' and 'endturn' are mutually exclusive. Use only 'endline' to specify the line number to end analysis, or 'endturn' to specify the speaker turn number to end analysis.")
    if startline is not None or endline is not None or startturn is not None or endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with line or turn parameters. Use 'runcount' alone or with either 'startline' and 'endline', or 'startturn' and 'endturn'.")
    
    with open(input_file, 'r') as file:
        lines = file.readlines()

    # Process lines and extract speaker turns
    speaker_turns = process_lines(lines)
    
    # Determine start and end points for analysis
    if startline is not None:
        start = startline
    elif startturn is not None:
        start = startturn
    else:
        start = 0

    if endline is not None:
        end = endline
    elif endturn is not None:
        end = endturn
    else:
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(speaker_turns))  # Updated this line to consider turns instead of lines

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(speaker_turns):  # Updated this line to consider turns instead of lines
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines
        print(f"Warning: The end point exceeded the length of the speaker turns. It has been adjusted to the last speaker turn: {end}")

    # Create locks for writing to files
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically
        futures = [executor.submit(threadlock, i, speaker_turns, output_file_full, output_file_content, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            pass


In [9]:
transcript = "2008-09-26-debate-preclean.txt"

In [ ]:
epl(input_file=transcript, 
                    output_file_full="2008-full.json", 
                    output_file_content="2008-content.json")

In [11]:
import pandas as pd
import json
import re

def get_output_dataframe(filename):
    with open(filename, 'r') as file:
        data = [json.loads(line) for line in file]

    df = pd.json_normalize(data)

    # Extract the scores and explanations from the 'choices' column
    df['content'] = df['choices'].apply(lambda x: x[0]['message']['content'])

    def extract_score(content, category):
        match = re.search(f'{category}: ([\d.]+)', content)
        return float(match.group(1)) if match else None

    def extract_explanation(content, category):
        match = re.search(f'{category}_Explanation: (.+?)(?=[EPL]_Explanation|$)', content)
        return match.group(1).strip() if match else None

    df['Ethos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Ethos'))
    df['Pathos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Pathos'))
    df['Logos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Logos'))

    df['E_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'E'))
    df['P_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'P'))
    df['L_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'L'))

    # Drop the original 'content' column
    df = df.drop(columns=['content'])

    return df

In [29]:
costs = get_output_dataframe("2008-full.json")
costs

,id,object,created,model,choices,usage.prompt_tokens,usage.completion_tokens,usage.total_tokens,Ethos_Score,Pathos_Score,Logos_Score,E_Explanation,P_Explanation,L_Explanation
0,chatcmpl-7PlKwhh30ZFF5peBCOXJ90G3H1wAl,chat.completion,1686374518,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",425,78,503,None,None,None,"Lehrer establishes a minimal level of ethos, a...",There is no appeal to emotions in this simple ...,"Logos is not present as it is a question, not ..."
1,chatcmpl-7PlKwS5f9L8oF9foi3qZwOrz5EzRp,chat.completion,1686374518,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",304,74,378,None,None,None,The phrase does not contribute to the speaker'...,The phrase does not evoke emotions or appeal t...,The phrase does not present a logical argument...
2,chatcmpl-7PlKx9B8YB7lKOyUMhu5ifp320EPn,chat.completion,1686374519,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",158,82,240,None,None,None,"The phrase ""MCCAIN: Sure."" relies slightly on ...","There is no emotional appeal in the phrase,",The phrase lacks any logical argument or evide...
3,chatcmpl-7PlKxAMz52jc2X28sScPXN95o56mV,chat.completion,1686374519,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",733,75,808,None,None,None,McCain provides some background and credibilit...,McCain appeals to listeners by mentioning the ...,McCain discusses a logical solution to reduce ...
4,chatcmpl-7PlKwLcTiTInhdnqgVOqpBGZp9Lr7,chat.completion,1686374518,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",437,95,532,None,None,None,McCain establishes credibility by referring to...,The emotional appeal comes from referring to P...,He provides logical arguments by addressing th...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,chatcmpl-7PlOGsgBqBVqjfOyhTfGpgOoerw42,chat.completion,1686374724,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",661,174,835,None,None,None,Lehrer establishes credibility as the moderato...,The question evokes emotions of fear and secur...,McCain provides logical reasoning by mentionin...
381,chatcmpl-7PlOHzM2zZQEGOtllCGAKXuQ9E27N,chat.completion,1686374725,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",436,140,576,None,None,None,McCain establishes credibility by referencing ...,McCain evokes fear of failure in Iraq and its ...,None
382,chatcmpl-7PlOHFG4wCOR2zrWGKeqWTIgaZ5Qa,chat.completion,1686374725,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",956,196,1152,None,None,None,"No appeal to credibility or character,","No appeal to emotions or values,",Presents logical arguments about the current s...
383,chatcmpl-7PlOJH4Yl6eoQXRW5AGNIke5zGtpN,chat.completion,1686374727,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",804,196,1000,None,None,None,The sentence has low Ethos as it doesn't estab...,Moderate Pathos due to directly addressing McC...,Logos is strong as Obama lists the consequence...


In [13]:
import pandas as pd

def get_total_cost(df):
    # Assuming df is the DataFrame obtained from the function process_jsonl_to_df

    # Calculate the total tokens
    total_prompt_tokens = df["usage.prompt_tokens"].sum()
    total_response_tokens = df["usage.completion_tokens"].sum()

    # Cost for 1k tokens
    prompt_token_price = 0.03  # This is the price per 1k prompt tokens
    response_token_price = 0.06  # This is the price per 1k response tokens

    # Calculate the total costs
    total_prompt_cost = (total_prompt_tokens / 1000) * prompt_token_price
    total_response_cost = (total_response_tokens / 1000) * response_token_price

    # Total cost
    total_cost = total_prompt_cost + total_response_cost

    # Create a summary dataframe
    summary = pd.DataFrame({
        "total_prompt_tokens": [total_prompt_tokens],
        "prompt_cost": [total_prompt_cost],
        "total_response_tokens": [total_response_tokens],
        "response_cost": [total_response_cost],
        "total_cost": [total_cost]
    })

    return summary

In [20]:
get_total_cost(costs)

,total_prompt_tokens,prompt_cost,total_response_tokens,response_cost,total_cost
0,165065,4.95195,50665,3.0399,7.99185


In [21]:
import pandas as pd
import re
import json
import numpy as np

def get_score_details(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    scores = []
    explanations = []

    # Regex patterns
    score_pattern = re.compile(r'Ethos: ([0-9.]+), Pathos: ([0-9.]+), Logos: ([0-9.]+)')
    explanation_pattern = re.compile(r'E_Explanation: (.*), P_Explanation: (.*), L_Explanation: (.*)')

    for line in lines:
        content = json.loads(line)['choices'][0]['message']['content']

        # Use regex to locate and extract scores and explanations
        score_match = score_pattern.findall(content)
        explanation_match = explanation_pattern.findall(content)

        # If there are multiple evaluations in a single "content" field, handle them separately
        for score, explanation in zip(score_match, explanation_match):
            scores.append(score)
            explanations.append(explanation)

    # Convert lists of tuples to dataframes
    scores_df = pd.DataFrame(scores, columns=['Ethos_Score', 'Pathos_Score', 'Logos_Score'])
    explanations_df = pd.DataFrame(explanations, columns=['E_Explanation', 'P_Explanation', 'L_Explanation'])

    # Convert score columns to numeric types
    for col in ['Ethos_Score', 'Pathos_Score', 'Logos_Score']:
        scores_df[col] = pd.to_numeric(scores_df[col])

    # Concatenate dataframes
    result_df = pd.concat([scores_df, explanations_df], axis=1)

    return result_df

# Usage
file_path = "2008-full.json"
result_df = get_score_details(file_path)


In [22]:
result_df

,Ethos_Score,Pathos_Score,Logos_Score,E_Explanation,P_Explanation,L_Explanation


In [26]:
import pandas as pd
import json
import re

def epl_json_to_df(filename):
    df_list = []
    with open(filename, 'r') as file:
        for line in file:
            item = json.loads(line)
            current_sentence = item.get("curr_turn", "")
            content = item.get("content", "")
            
            # Extract the scores using regex
            ethos_score = re.search('E: (\d.\d)', content)
            pathos_score = re.search('P: (\d.\d)', content)
            logos_score = re.search('L: (\d.\d)', content)
            
            # Extract the explanations using regex
            ethos_explanation = re.search('E_Explanation: (.*), P_Explanation', content)
            pathos_explanation = re.search('P_Explanation: (.*), L_Explanation', content)
            logos_explanation = re.search('L_Explanation: (.*)', content)

            # Create a dictionary for the current item
            df_dict = {
                "current_sentence": current_sentence,
                "Ethos": ethos_score.group(1) if ethos_score else None,
                "Pathos": pathos_score.group(1) if pathos_score else None,
                "Logos": logos_score.group(1) if logos_score else None,
                "E_Explanation": ethos_explanation.group(1) if ethos_explanation else None,
                "P_Explanation": pathos_explanation.group(1) if pathos_explanation else None,
                "L_Explanation": logos_explanation.group(1) if logos_explanation else None
            }
            
            df_list.append(df_dict)

    df = pd.DataFrame(df_list)
    return df


In [28]:
scores = epl_json_to_df('2008-content.json')
scores


,current_sentence,Ethos,Pathos,Logos,E_Explanation,P_Explanation,L_Explanation
0,"LEHRER: Are you going to vote for the plan, Se...",1.0,0.0,0.0,"Lehrer establishes a minimal level of ethos, a...",There is no appeal to emotions in this simple ...,"Logos is not present as it is a question, not ..."
1,LEHRER: Say it directly to him.\n,0.0,0.0,0.0,The phrase does not contribute to the speaker'...,The phrase does not evoke emotions or appeal t...,The phrase does not present a logical argument...
2,MCCAIN: Sure.\n,1.0,0.0,0.0,"The phrase ""MCCAIN: Sure."" relies slightly on ...",There is no emotional appeal in the phrase,The phrase lacks any logical argument or evide...
3,"LEHRER: All right, let’s go to the next lead q...",5.5,4.5,8.5,McCain provides some background and credibilit...,McCain appeals to listeners by mentioning the ...,McCain discusses a logical solution to reduce ...
4,MCCAIN: Sure. But — but let me — let me point ...,7.2,6.1,8.0,McCain establishes credibility by referring to...,The emotional appeal comes from referring to P...,He provides logical arguments by addressing th...
...,...,...,...,...,...,...,...
380,LEHRER: This is the last — last lead question....,7.5,4.5,6.5,Lehrer establishes credibility as the moderato...,The question evokes emotions of fear and secur...,"The question is logical, asking about the prob..."
381,"MCCAIN: But in the case of missile defense, Se...",7.9,7.8,6.5,McCain establishes credibility by referencing ...,McCain evokes fear of failure in Iraq and its ...,McCain cites General Petraeus's statements and...
382,"LEHRER: Two minutes, Senator Obama.\n",0.0,0.0,0.0,No appeal to credibility or character,No appeal to emotions or values,No logical argument or evidence presented.
383,LEHRER: You see the same connections that Sena...,2.1,3.2,4.7,The sentence has low Ethos as it doesn't estab...,Moderate Pathos due to directly addressing McC...,"High Logos as questions logic, but does not in..."
